# Train the 32-word ASL model

This notebook turns the videos in `training/new_videos` (team) and `training/raw_videos` (online) into the model the Jetson runs.
It goes in order: **find the videos -> pull keypoints out of them -> hand check (throw out clips where the hands can't be seen) -> split -> train -> test -> export**.

## Directions (PC, recommended)
1. Open a terminal in the `training` folder of the repo and run these **once**:
   ```
   py -3.10 -m venv .venv
   .venv\Scripts\activate
   pip install tensorflow==2.15.1 numpy==1.26.4 protobuf==4.25.9 opencv-python-headless==4.10.0.84 matplotlib ipykernel attrs absl-py flatbuffers
   pip install --no-deps mediapipe==0.10.18
   ```
   (In Git Bash use `source .venv/Scripts/activate` instead of the second line.)
2. Open `train_model.ipynb` in Cursor or VS Code. Top right: **Select Kernel -> Python Environments -> `.venv`**.
3. Skip step 0 (that one is only for Colab).
4. Run the cells **one at a time from step 1 down**. Each one prints what it did, and stops with a message that starts with `PROBLEM:` if something is wrong.
5. The slow part is **step 4 (keypoints)**: about 20-40 minutes the first time for ~2000 clips. It saves its work in `training/_keypoints/`, so running it again (or after a crash) only does the new clips.
6. **Look at step 5 (hand check)** before training. It tells you which clips were thrown out and why, with pictures.
7. Step 8 trains (about 15-40 minutes on a CPU). Step 9 tests it. Step 10 saves the model to `training/output/`.
8. To use it on the Jetson, set `COPY_TO_JETSON = True` in step 1 (or copy the `training/output/<MODEL_NAME>/` folder to `jetson-code/models/`), then start the Jetson with `MODEL_PATH=models/<MODEL_NAME>/<MODEL_NAME>.h5`. The word list and settings are read from `model_meta.json` next to the model.

## Directions (Google Colab, optional)
Colab downloads the videos from GitHub itself, so you do **not** upload videos to Drive. Only this notebook and the results go in Drive.
1. Upload `train_model.ipynb` to Google Drive and open it with Colab.
2. If the repo is private: left bar **key icon (Secrets) -> Add new secret**, name `GITHUB_TOKEN`, value = a GitHub token that can read the repo, and turn on "Notebook access".
3. Run step 0. When it says so, **Runtime -> Restart session**, then run step 0 again (it will say it's ready).
4. Run step 1 down, like on the PC. Keypoints and results are saved in `MyDrive/ASL_training/`, so a disconnect does not lose the slow work.

## Good to know
- **Clip names.** Team clips must be `<word>_<name>_<number>.mp4` inside `new_videos/<word>/`. Online clips can have any name inside `raw_videos/<word>/`.
- **Settings.** Everything you might change is in step 1. The defaults are what was tested.
- **Same steps live.** Videos are read at 15 frames per second and in 30-frame windows, exactly like the Jetson (30 fps camera, every 2nd frame).
- **Retraining.** Re-running is cheap after the first time. Only new or changed clips get keypoints again.

In [ ]:
#@title 0. Colab only: install (skip this on the PC)
import sys, subprocess, importlib

PINS = ["numpy==1.26.4", "protobuf==4.25.9", "opencv-python-headless==4.10.0.84", "tensorflow==2.15.1",
        "attrs", "absl-py", "flatbuffers", "matplotlib"]

def _have(name, version):
    try:
        return importlib.import_module(name).__version__.startswith(version)
    except Exception:
        return False

if "google.colab" not in sys.modules:
    print("Not in Colab: nothing to do here. Go to step 1.")
elif _have("numpy", "1.26") and _have("mediapipe", "0.10.18") and _have("tensorflow", "2.15"):
    print("Ready. Go to step 1.")
else:
    print("Installing (2-3 minutes)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "mediapipe==0.10.18"], check=True)
    print("\nDone. Now: Runtime -> Restart session, then run this cell again.")

In [ ]:
#@title 1. Settings
# Words: None = every word folder (words with fewer than MIN_CLIPS_PER_WORD usable clips are left out).
# Or a list, e.g. ["help", "pain", "yes"].
WORDS = None
SOURCES = {"team": "new_videos", "online": "raw_videos"}   # name -> folder inside training/
MIN_CLIPS_PER_WORD = 20
VIDEO_EXTS = (".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v")

# Hand check (step 5). A frame is "usable" when:
HAND_POINTS_INSIDE = 18   # ...at least one hand is found with this many of its 21 points inside the picture
                          #     (or a pose wrist is visible inside the picture — fallback when the hand model misses),
VISIBILITY_MIN = 0.5      # ...both shoulders are seen (visibility >= this),
BODY_POINTS_MIN = 5       # ...and this many of 7 upper-body points (nose, shoulders, elbows, wrists) are seen.
ACTIVE_USABLE_MIN = 0.70  # keep a clip if this share of frames WHILE SIGNING are usable
CLIP_USABLE_MIN = 0.50    # ...and this share of the whole clip (rest poses included) is usable
MIN_SEGMENT = 24          # a failed clip can be rescued if it has a usable stretch this long (frames at 15 fps)
GAP_TOLERANCE = 3         # short dropouts (frames) allowed inside a rescued stretch
ACTIVE_MOVE = 0.35        # wrist movement (in shoulder widths) away from the rest position that counts as signing
ACTIVE_PAD = 5            # frames added before/after the signing part
TWO_HAND_FLAG = 0.30      # for two-handed words: flag clips where both hands are seen less than this

# Timing: must match the Jetson (30 fps camera, every 2nd frame -> 15 fps, 30-frame window = 2 s)
SAMPLE_FPS = 15
WINDOW = 30
MAX_SIDE = 640            # frames are shrunk so the long side is at most this before MediaPipe

# Split
SEED = 42
ONLINE_TEST_FRAC = 0.20   # share of online clips (by source video) kept for the final test
VAL_FRAC = 0.15
HOLD_OUT_PERSON = None    # e.g. "aaron": test only on that person (needs words recorded by 2+ team members)

# Training windows and augmentation
WINDOWS_PER_CLIP = 8
LABEL_COVERAGE = 0.6      # a window counts as the word if it holds this share of the signing part
IDLE_CLASS = True         # add "_idle" (resting / between signs) so the model can say "no word"
TRANSITIONS = True        # add windows that run from the end of one sign into the next
TRANSITION_SHARE = 0.25
AUG_ROTATE_DEG = 15
AUG_SCALE = 0.25
AUG_SHIFT = 0.10
AUG_SQUASH = 0.15         # squeezes the person sideways a bit (looks like turning)
AUG_NOISE = 0.004
AUG_SPEED = 0.20          # signs 20% faster or slower
LOWER_BODY_DROPOUT = 0.30 # pretend the legs/hips are out of the picture (upper-body framing)
MIRROR_PROB = 0.5         # left-handed signers; never for the words below
NO_MIRROR_WORDS = ("left", "right")

# Training
EPOCHS = 200
BATCH = 64
PATIENCE = 25
LEARNING_RATE = 1e-3

# Live rules written into model_meta.json (also used by the "Jetson replay" test in step 9)
THRESHOLD = 0.70
STABLE_N = 8
MIN_HAND_FRAC = 0.6
REPEAT_RESET = 15
FRAME_SIZE = [640, 360]   # the Jetson crops/resizes camera frames to this

# Run
WORKERS = 6               # clips processed at the same time in step 4
MODEL_NAME = "asl_32word_test1"
COPY_TO_JETSON = True     # True: also copy the model into jetson-code/models/<MODEL_NAME>/
QUICK_TEST = 0            # > 0: only use this many clips per word and source (a fast trial run)
TRAINING_DIR = None       # path to the training folder, if step 2 can't find it

# Colab only
REPO = "SJSU-CMPE-195/group-project-team-narm"
BRANCH = "new-model-training"
DRIVE_FOLDER = "ASL_training"
print("Settings saved.")

In [ ]:
#@title 2. Setup and check
import os, sys, json, time, re, math, hashlib, subprocess, importlib, random, shutil, csv
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np

IN_COLAB = "google.colab" in sys.modules

def problem(msg):
    raise RuntimeError("PROBLEM: " + msg)

if sys.version_info[:2] not in ((3, 10), (3, 11)):
    problem(f"Python {sys.version.split()[0]}. Use Python 3.10 (PC) or Colab. See the directions at the top.")
if not np.__version__.startswith("1."):
    problem(f"numpy {np.__version__}; MediaPipe needs numpy 1.x. PC: pip install numpy==1.26.4. Colab: run step 0.")
try:
    import cv2
    import mediapipe as mp
    import tensorflow as tf
except Exception as e:
    problem(f"Import failed ({e}). PC: run the pip lines from the directions. Colab: run step 0 and restart.")
if mp.__version__ != "0.10.18" or not tf.__version__.startswith("2.15"):
    print(f"Warning: mediapipe {mp.__version__}, tensorflow {tf.__version__} (tested: 0.10.18 and 2.15.1).")

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd).replace(os.environ.get("_TOK", "\0"), "***"))
    subprocess.run([str(c) for c in cmd], check=True, **kw)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/repo")
    if not (REPO_DIR / "training").is_dir():
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            pass
        if token:
            os.environ["_TOK"] = token
        url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
        if subprocess.run(["git", "lfs", "version"], capture_output=True).returncode != 0:
            run(["apt-get", "-qq", "install", "git-lfs"])
        env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
        try:
            run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "-b", BRANCH, url, REPO_DIR], env=env)
        except subprocess.CalledProcessError:
            problem("Could not download the repo. If it is private, add the GITHUB_TOKEN secret (see the directions).")
        run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "training/new_videos", "training/raw_videos", "jetson-code"], env=env)
        run(["git", "-C", REPO_DIR, "lfs", "install", "--local"])
        run(["git", "-C", REPO_DIR, "lfs", "pull"])
    TRAIN_DIR = REPO_DIR / "training"
    WORK_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER
else:
    def _find_training():
        if TRAINING_DIR:
            return Path(TRAINING_DIR).resolve()
        here = Path.cwd().resolve()
        for p in (here, *here.parents):
            for cand in (p, p / "training"):
                if (cand / "new_videos").is_dir() and (cand.parent / "jetson-code").is_dir():
                    return cand
        return None
    TRAIN_DIR = _find_training()
    if TRAIN_DIR is None:
        problem("Could not find the training folder. Open this notebook from the repo, or set TRAINING_DIR in step 1.")
    WORK_DIR = TRAIN_DIR

JETSON_DIR = TRAIN_DIR.parent / "jetson-code"
CACHE_DIR = WORK_DIR / "_keypoints"
OUT_DIR = WORK_DIR / "output" / MODEL_NAME
for d in (CACHE_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
if str(JETSON_DIR) not in sys.path:
    sys.path.insert(0, str(JETSON_DIR))
import asl_features as F
importlib.reload(F)
if F.FEATURE_VERSION != "pose-hands-v2" or not hasattr(F, "normalize"):
    problem(f"jetson-code/asl_features.py is {F.FEATURE_VERSION}; this notebook needs pose-hands-v2. Pull the latest code.")

random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print(f"Python {sys.version.split()[0]} | numpy {np.__version__} | mediapipe {mp.__version__} | "
      f"tensorflow {tf.__version__} | opencv {cv2.__version__}")
print(f"Videos:    {TRAIN_DIR}")
print(f"Keypoints: {CACHE_DIR}")
print(f"Results:   {OUT_DIR}")
print(f"Features:  {F.FEATURE_VERSION} ({F.FEATURE_DIM} values per frame)")
print("GPU:", tf.config.list_physical_devices("GPU") or "none (CPU is fine for this model)")

In [ ]:
#@title 3. Find the videos
def file_id(path, size):
    h = hashlib.sha1(str(size).encode())
    with open(path, "rb") as f:
        h.update(f.read(1 << 18))
    return h.hexdigest()[:16]

def online_group(stem):
    """Clips cut from the same source video share a group (they must stay on the same side of the split)."""
    s = stem.lower()
    m = re.search(r"media_([a-z0-9_-]{11})_", s)
    if m:
        return "yt:" + m.group(1)
    s = s.replace("(online-video-cutter.com)", "")
    s = re.sub(r"\s*\(\d+\)\s*$", "", s.strip())
    s = re.sub(r"^\d+[_ -]+", "", s)
    return s or stem.lower()

clips, problems_found, seen_ids = [], [], {}
lfs_missing, duplicates = [], []
for source, folder in SOURCES.items():
    root = TRAIN_DIR / folder
    if not root.is_dir():
        problem(f"Missing folder {root}")
    for wdir in sorted(p for p in root.iterdir() if p.is_dir()):
        files = sorted(f for f in wdir.rglob("*") if f.is_file() and f.suffix.lower() in VIDEO_EXTS)
        if QUICK_TEST:
            files = files[:QUICK_TEST]
        for f in files:
            size = f.stat().st_size
            if size < 2048 and f.read_bytes()[:30].startswith(b"version https://git-lfs"):
                lfs_missing.append(f); continue
            fid = file_id(f, size)
            rel = f.relative_to(TRAIN_DIR).as_posix()
            if fid in seen_ids:
                duplicates.append((rel, seen_ids[fid])); continue
            seen_ids[fid] = rel
            c = dict(word=wdir.name, source=source, path=f, rel=rel, id=fid, size=size)
            if source == "team":
                parts = f.stem.rsplit("_", 2)
                if len(parts) == 3 and parts[0] == wdir.name and parts[2].isdigit():
                    c.update(person=parts[1].lower(), take=int(parts[2]), group=f"team:{parts[1].lower()}")
                else:
                    problems_found.append(f"{rel}: name should be {wdir.name}_<name>_<number>")
                    c.update(person="unknown", take=len(clips), group=f"team:unknown:{f.stem}")
            else:
                c.update(person="online", take=0, group=f"online:{wdir.name}:{online_group(f.stem)}")
            clips.append(c)

if lfs_missing:
    problem(f"{len(lfs_missing)} videos are Git LFS placeholders, not real videos (e.g. {lfs_missing[0]}). "
            "Run `git lfs pull` in the repo, then run this step again.")
if WORDS:
    missing = [w for w in WORDS if not any(c["word"] == w for c in clips)]
    if missing:
        problem(f"No videos for {missing}")
    clips = [c for c in clips if c["word"] in WORDS]

all_words = sorted({c["word"] for c in clips})
people = sorted({c["person"] for c in clips if c["source"] == "team"})
print(f"{len(clips)} clips, {len(all_words)} words. Team signers: {', '.join(people)}")
print(f"Formats: {dict(Counter(c['path'].suffix.lower() for c in clips))}")
if duplicates:
    print(f"Skipped {len(duplicates)} exact duplicate files (same video twice), e.g. {duplicates[0][0]} = {duplicates[0][1]}")
for p in problems_found[:10]:
    print("Name problem:", p)

head = f"{'word':16}" + "".join(f"{p[:7]:>8}" for p in people) + f"{'online':>8}{'groups':>8}{'total':>7}"
print("\n" + head + "\n" + "-" * len(head))
for w in all_words:
    wc = [c for c in clips if c["word"] == w]
    row = f"{w:16}" + "".join(f"{sum(c['person'] == p for c in wc) or '':>8}" for p in people)
    on = [c for c in wc if c["source"] == "online"]
    row += f"{len(on):>8}{len({c['group'] for c in on}):>8}{len(wc):>7}"
    print(row)
single = [w for w in all_words if len({c["person"] for c in clips if c["word"] == w and c["source"] == "team"}) == 1]
if single:
    print(f"\nNote: {len(single)} words are recorded by only ONE team signer. The online clips are what stop the model "
          "from learning 'who is signing' instead of 'what is signed'; step 9 checks this.")

In [ ]:
#@title 3b. Quick look: are the keypoints right? (4 random clips)
import matplotlib.pyplot as plt

POSE_LINES = [(11, 12), (11, 13), (13, 15), (12, 14), (14, 16), (11, 23), (12, 24), (23, 24),
              (23, 25), (25, 27), (24, 26), (26, 28)]
HAND_LINES = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11), (11, 12),
              (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]

def shrink(frame):
    h, w = frame.shape[:2]
    s = MAX_SIDE / max(h, w)
    return cv2.resize(frame, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA) if s < 1 else frame

def draw(frame_rgb, raw):
    img = frame_rgb.copy()
    h, w = img.shape[:2]
    pose = raw[F.POSE_SLICE].reshape(33, 4)
    px = lambda p: (int(p[0] * w), int(p[1] * h))
    for a, b in POSE_LINES:
        if pose[a, 3] >= VISIBILITY_MIN and pose[b, 3] >= VISIBILITY_MIN:
            cv2.line(img, px(pose[a]), px(pose[b]), (255, 255, 255), 2)
    for color, sl in (((255, 80, 80), F.HAND_A_SLICE), ((80, 160, 255), F.HAND_B_SLICE)):
        hand = raw[sl].reshape(21, 3)
        if hand.any():
            for a, b in HAND_LINES:
                cv2.line(img, px(hand[a]), px(hand[b]), color, 2)
    return img

def frame_at(path, k, fps):
    cap = cv2.VideoCapture(str(path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(int(round(k * fps / SAMPLE_FPS)), 0))
    ok, frame = cap.read()
    cap.release()
    return cv2.cvtColor(shrink(frame), cv2.COLOR_BGR2RGB) if ok else None

rng_look = random.Random(SEED)
look = rng_look.sample([c for c in clips if c["source"] == "team"], 2) + \
       rng_look.sample([c for c in clips if c["source"] == "online"] or clips, 2)
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
det = F.Detector(model_complexity=0)
for ax, c in zip(axes, look):
    cap = cv2.VideoCapture(str(c["path"]))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    cap.set(cv2.CAP_PROP_POS_FRAMES, n // 2)
    ok, frame = cap.read()
    cap.release()
    ax.axis("off")
    if not ok:
        ax.set_title(f"{c['rel']}\n(could not read)", fontsize=8); continue
    rgb = cv2.cvtColor(shrink(frame), cv2.COLOR_BGR2RGB)
    for _ in range(3):  # a few passes so tracking settles
        raw = det.process(rgb)
    feats = F.normalize(raw, rgb.shape[1] / rgb.shape[0])
    ax.imshow(draw(rgb, raw))
    ax.set_title(f"{c['rel']}\nhands: {int(raw[F.HAND_A_SLICE].any()) + int(raw[F.HAND_B_SLICE].any())}, "
                 f"features {feats.min():.1f}..{feats.max():.1f}", fontsize=8)
det.close()
plt.tight_layout(); plt.show()
print("White = body, red/blue = the two hands. If these look wrong, stop here and say so.")

In [ ]:
#@title 4. Pull keypoints out of every video (slow the first time, then cached)
EXTRACT_TAG = f"v2-{SAMPLE_FPS}fps-{MAX_SIDE}px-mc0-mp{mp.__version__}"

def cache_file(c):
    return CACHE_DIR / f"{c['id']}_{hashlib.sha1(EXTRACT_TAG.encode()).hexdigest()[:6]}.npz"

def extract(c):
    """Every video -> raw keypoints at 15 fps. Frames are turned upright (phone videos) and shrunk first."""
    cap = cv2.VideoCapture(str(c["path"]))
    if not cap.isOpened():
        raise RuntimeError("cannot open")
    fps = cap.get(cv2.CAP_PROP_FPS)
    fps = fps if 1 <= fps <= 240 else 30.0
    det = F.Detector(model_complexity=0)
    out, i, ticks, size = [], 0, 0, None
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            due = int(math.floor(i / fps * SAMPLE_FPS + 1e-6)) + 1 - ticks  # samples due by this frame
            i += 1
            if due <= 0:
                continue
            ticks += due
            if size is None:
                size = frame.shape[1], frame.shape[0]
            raw = det.process(cv2.cvtColor(shrink(frame), cv2.COLOR_BGR2RGB))
            out.extend([raw] * due)
    finally:
        det.close()
        cap.release()
    if not out:
        raise RuntimeError("no frames")
    return dict(raw=np.asarray(out, np.float32), aspect=np.float32(size[0] / size[1]), fps=np.float32(fps),
                src_frames=np.int32(i), width=np.int32(size[0]), height=np.int32(size[1]))

def load_or_extract(c):
    f = cache_file(c)
    if f.exists():
        try:
            with np.load(f) as z:
                return {k: z[k] for k in z.files}, True
        except Exception:
            pass
    d = extract(c)
    tmp = f.with_suffix(".tmp.npz")
    np.savez_compressed(tmp, **d)
    os.replace(tmp, f)
    return d, False

t0, done, fresh, failed = time.time(), 0, 0, []
todo = [c for c in clips if "raw" not in c]
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(load_or_extract, c): c for c in todo}
    for fut in as_completed(futures):
        c = futures[fut]
        done += 1
        try:
            d, cached = fut.result()
            c.update(raw=d["raw"], aspect=float(d["aspect"]), fps=float(d["fps"]),
                     width=int(d["width"]), height=int(d["height"]))
            fresh += not cached
        except Exception as e:
            failed.append((c["rel"], str(e)))
        if done % 50 == 0 or done == len(todo):
            el = time.time() - t0
            eta = el / done * (len(todo) - done)
            print(f"  {done}/{len(todo)} clips ({fresh} new) | {el / 60:.1f} min | about {eta / 60:.1f} min left", flush=True)

clips = [c for c in clips if "raw" in c]
print(f"\nKeypoints ready for {len(clips)} clips ({fresh} newly extracted, {len(clips) - fresh} from the cache).")
for rel, err in failed:
    print(f"  could not read {rel}: {err}")
fps_seen = Counter(round(c["fps"]) for c in clips)
portrait = sum(c["height"] > c["width"] for c in clips)
print(f"Frame rates: {dict(sorted(fps_seen.items()))} | portrait clips: {portrait} | "
      f"length at 15 fps: {min(len(c['raw']) for c in clips)}-{max(len(c['raw']) for c in clips)} frames")

In [ ]:
#@title 5. HAND CHECK: keep only clips where the hands (and upper body) can be seen
UPPER_BODY = [0, 11, 12, 13, 14, 15, 16]

def frame_checks(raw):
    """Usable = upper body seen AND at least one hand in view.

    Prefer the hand model (21 points). If it misses (overlap / blur), fall back to a
    pose wrist that is visible and inside the picture — same rule as \"hands are there\".
    """
    pose = raw[:, F.POSE_SLICE].reshape(-1, 33, 4)
    def hand_ok(sl):
        h = raw[:, sl].reshape(-1, 21, 3)
        inside = (h[..., 0] >= 0) & (h[..., 0] <= 1) & (h[..., 1] >= 0) & (h[..., 1] <= 1)
        return np.any(h != 0, axis=(1, 2)) & (inside.sum(1) >= HAND_POINTS_INSIDE)
    a, b = hand_ok(F.HAND_A_SLICE), hand_ok(F.HAND_B_SLICE)
    vis = pose[..., 3] >= VISIBILITY_MIN
    # Pose wrists: 15 = MediaPipe left (= signer's right on unmirrored frames), 16 = right
    def wrist_in(idx):
        xy = pose[:, idx, :2]
        return vis[:, idx] & (xy[:, 0] >= 0) & (xy[:, 0] <= 1) & (xy[:, 1] >= 0) & (xy[:, 1] <= 1)
    hand_any = a | b | wrist_in(15) | wrist_in(16)
    hand_both = (a | wrist_in(15)) & (b | wrist_in(16))
    body = vis[:, 11] & vis[:, 12] & (vis[:, UPPER_BODY].sum(1) >= BODY_POINTS_MIN)
    return hand_any & body, hand_both, hand_any, body

def active_part(raw, aspect):
    """Where the signing happens: wrists moved away from where they were at the start and end of the clip."""
    T = len(raw)
    pose = raw[:, F.POSE_SLICE].reshape(-1, 33, 4)
    xy = pose[:, :, :2] * np.array([aspect, 1.0])
    width = np.linalg.norm(xy[:, 11] - xy[:, 12], axis=1)
    ok = width > 1e-3
    if ok.sum() < 6:
        return 0, T, False
    wr = xy[:, [15, 16]]
    okw = wr[ok]
    start, end = np.median(okw[:3], axis=0), np.median(okw[-3:], axis=0)
    d = np.minimum(np.linalg.norm(wr - start, axis=2).max(1), np.linalg.norm(wr - end, axis=2).max(1))
    d = np.where(ok, d / np.median(width[ok]), 0)
    idx = np.where(d > ACTIVE_MOVE)[0]
    if len(idx) < 3:
        return 0, T, False
    return max(0, idx[0] - ACTIVE_PAD), min(T, idx[-1] + 1 + ACTIVE_PAD), True

def longest_run(mask, gap):
    best, s, last, miss = (0, 0), None, None, 0
    for i, m in enumerate(mask):
        if m:
            if s is None:
                s = i
            last, miss = i, 0
        elif s is not None:
            miss += 1
            if miss > gap:
                best = max(best, (s, last + 1), key=lambda r: r[1] - r[0])
                s, miss = None, 0
    if s is not None:
        best = max(best, (s, last + 1), key=lambda r: r[1] - r[0])
    return best

for c in clips:
    usable, both, anyhand, body = frame_checks(c["raw"])
    a0, a1, found = active_part(c["raw"], c["aspect"])
    act_u, clip_u = usable[a0:a1].mean(), usable.mean()
    c.update(act=(a0, a1), act_found=found, act_usable=float(act_u), clip_usable=float(clip_u),
             both_share=float(both[a0:a1].mean()), hand_share=float(anyhand.mean()), body_share=float(body.mean()))
    if act_u >= ACTIVE_USABLE_MIN and clip_u >= CLIP_USABLE_MIN:
        c.update(verdict="keep", seg=(0, len(usable)))
        continue
    s, e = longest_run(usable, GAP_TOLERANCE)
    if e - s >= MIN_SEGMENT and usable[s:e].mean() >= ACTIVE_USABLE_MIN:
        c.update(verdict="rescue", seg=(s, e))
        lo, hi = max(a0, s), min(a1, e)
        c["act"] = (lo, hi) if hi - lo >= 6 else (s, e)
    else:
        c["verdict"] = "drop"
        if act_u >= ACTIVE_USABLE_MIN:
            c["why"] = "hands/body missing in most of the clip (outside the signing part)"
        elif c["hand_share"] < ACTIVE_USABLE_MIN and c["body_share"] >= ACTIVE_USABLE_MIN:
            c["why"] = "hands not found (or cut off by the picture edge)"
        elif c["body_share"] < ACTIVE_USABLE_MIN:
            c["why"] = "shoulders/upper body not seen"
        else:
            c["why"] = "hands and body rarely seen at the same time"

two_handed = {w for w in all_words
              if np.median([c["both_share"] for c in clips if c["word"] == w and c["verdict"] != "drop"] or [0]) > 0.5}
for c in clips:
    c["two_hand_flag"] = c["word"] in two_handed and c["verdict"] != "drop" and c["both_share"] < TWO_HAND_FLAG

V = lambda w, s, v: sum(c["word"] == w and c["source"] == s and c["verdict"] == v for c in clips)
head = f"{'word':16}{'team keep/rescue/drop':>23}{'online keep/rescue/drop':>25}{'usable':>8}  note"
print(head + "\n" + "-" * (len(head) + 10))
usable_words = []
for w in all_words:
    n = sum(c["word"] == w and c["verdict"] != "drop" for c in clips)
    note = "TWO-HANDED" if w in two_handed else ""
    if n < MIN_CLIPS_PER_WORD:
        note += f"  LEFT OUT (< {MIN_CLIPS_PER_WORD} usable clips)"
    else:
        usable_words.append(w)
    print(f"{w:16}{V(w, 'team', 'keep'):>11}/{V(w, 'team', 'rescue'):>3}/{V(w, 'team', 'drop'):>3}"
          f"{V(w, 'online', 'keep'):>17}/{V(w, 'online', 'rescue'):>3}/{V(w, 'online', 'drop'):>3}{n:>8}  {note}")

tot = Counter(c["verdict"] for c in clips)
print(f"\nTotal: {tot['keep']} kept, {tot['rescue']} rescued (trimmed to their good part), {tot['drop']} dropped "
      f"of {len(clips)}.")
for src in SOURCES:
    t = Counter(c["verdict"] for c in clips if c["source"] == src)
    print(f"  {src}: {t['keep']} kept, {t['rescue']} rescued, {t['drop']} dropped")
only_clip = sum(c["verdict"] != "keep" and c["act_usable"] >= ACTIVE_USABLE_MIN for c in clips)
print(f"  {only_clip} clips missed 'keep' only because of the whole-clip rule (CLIP_USABLE_MIN = {CLIP_USABLE_MIN}).")
print(f"  {sum(not c['act_found'] for c in clips)} clips had no clear rest position, so the whole clip counts as signing.")
why = Counter(c["why"] for c in clips if c["verdict"] == "drop")
for k, v in why.most_common():
    print(f"  dropped because {k}: {v}")
flagged = [c for c in clips if c["two_hand_flag"]]
if flagged:
    print(f"  {len(flagged)} clips of two-handed words show both hands less than {TWO_HAND_FLAG:.0%} of the time "
          f"(kept, but worth a look): {', '.join(c['rel'] for c in flagged[:6])}{' ...' if len(flagged) > 6 else ''}")
ACTIONS = sorted(usable_words) + (["_idle"] if IDLE_CLASS else [])
print(f"\nThe model will learn {len(usable_words)} words" + (" + _idle (no sign)." if IDLE_CLASS else "."))

with open(OUT_DIR / "hand_check.csv", "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["clip", "word", "source", "verdict", "why", "usable_while_signing", "usable_whole_clip",
                 "both_hands_while_signing", "signing_frames", "kept_frames"])
    for c in sorted(clips, key=lambda c: c["rel"]):
        wr.writerow([c["rel"], c["word"], c["source"], c["verdict"], c.get("why", ""), f"{c['act_usable']:.2f}",
                     f"{c['clip_usable']:.2f}", f"{c['both_share']:.2f}", f"{c['act'][0]}-{c['act'][1]}",
                     f"{c['seg'][0]}-{c['seg'][1]}" if "seg" in c else ""])
print(f"Every clip's result: {OUT_DIR / 'hand_check.csv'}")

worst = sorted([c for c in clips if c["verdict"] != "keep"], key=lambda c: c["act_usable"])[:8]
if worst:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    for ax in axes.flat:
        ax.axis("off")
    for ax, c in zip(axes.flat, worst):
        usable = frame_checks(c["raw"])[0]
        a0, a1 = c["act"]
        bad = [k for k in range(a0, a1) if not usable[k]] or [(a0 + a1) // 2]
        k = bad[len(bad) // 2]
        img = frame_at(c["path"], k, c["fps"])
        if img is not None:
            ax.imshow(draw(img, c["raw"][k]))
        ax.set_title(f"{c['rel']}\n{c['verdict']}: {c['act_usable']:.0%} usable while signing", fontsize=8)
    plt.suptitle("Worst clips (a frame that failed the check)")
    plt.tight_layout(); plt.show()

In [ ]:
#@title 6. Split into train / validation / test (no source video ends up on two sides)
rng = np.random.default_rng(SEED)
data = [c for c in clips if c["verdict"] != "drop" and c["word"] in usable_words]
for c in data:
    c["split"] = "train"

team_split_note = "interleaved takes (every 7th take tests, every 7th validates)"
if HOLD_OUT_PERSON:
    for c in data:
        if c["source"] == "team" and c["person"] == HOLD_OUT_PERSON.lower():
            c["split"] = "test"
    team_split_note = f"all of {HOLD_OUT_PERSON}'s clips are the test"
else:
    by = defaultdict(list)
    for c in data:
        if c["source"] == "team":
            by[(c["word"], c["person"])].append(c)
    for lst in by.values():
        for i, c in enumerate(sorted(lst, key=lambda c: c["take"])):
            c["split"] = "test" if i % 7 == 3 else "val" if i % 7 == 6 else "train"

weak_online = []
for w in usable_words:
    on = [c for c in data if c["word"] == w and c["source"] == "online"]
    groups = defaultdict(list)
    for c in on:
        groups[c["group"]].append(c)
    keys = sorted(groups)
    rng.shuffle(keys)
    n = len(on)
    target = {"test": ONLINE_TEST_FRAC * n, "val": VAL_FRAC * n}
    filled = {"test": 0, "val": 0}
    if len(keys) >= 3:
        # Small source videos first, so one big source can't swallow the whole test set.
        for k in sorted(keys, key=lambda k: len(groups[k]) > max(target["test"], 1)):
            part = "train"
            for p in ("test", "val"):
                if filled[p] < target[p] and filled[p] + len(groups[k]) <= 1.5 * target[p] + 1:
                    part = p
                    break
            for c in groups[k]:
                c["split"] = part
            if part != "train":
                filled[part] += len(groups[k])
    if len(keys) < 3 or filled["test"] == 0 or sum(c["split"] == "train" for c in on) < n / 2:
        for c in on:
            c["split"] = "train"
        weak_online.append(w)
        for i, c in enumerate(sorted(on, key=lambda c: c["rel"])):
            c["split"] = "test" if i % 5 == 2 else "val" if i % 7 == 5 else "train"

sides = defaultdict(set)
for c in data:
    if c["source"] == "online" and c["word"] not in weak_online:
        sides[c["group"]].add(c["split"])
leaks = [g for g, s in sides.items() if len(s) > 1]
if leaks:
    problem(f"Split leak: {leaks[:3]}")

print(f"Team: {team_split_note}. Online: whole source videos go to one side ({ONLINE_TEST_FRAC:.0%} test).")
if weak_online:
    print(f"Note: {', '.join(weak_online)} have too few separate online source videos, so their online clips are "
          "split one by one (their online test score is less trustworthy).")
print(f"\n{'word':16}{'train':>7}{'val':>6}{'test':>6}   (team test / online test)")
for w in usable_words:
    wc = [c for c in data if c["word"] == w]
    cnt = Counter(c["split"] for c in wc)
    tt = sum(c["split"] == "test" and c["source"] == "team" for c in wc)
    print(f"{w:16}{cnt['train']:>7}{cnt['val']:>6}{cnt['test']:>6}   ({tt} / {cnt['test'] - tt})")
cnt = Counter(c["split"] for c in data)
print(f"{'TOTAL':16}{cnt['train']:>7}{cnt['val']:>6}{cnt['test']:>6}")
with open(OUT_DIR / "split.csv", "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["clip", "word", "source", "group", "split"])
    for c in sorted(data, key=lambda c: c["rel"]):
        wr.writerow([c["rel"], c["word"], c["source"], c["group"], c["split"]])

In [ ]:
#@title 7. Build training windows (+ augmentation, _idle, sign-to-sign transitions)
LABEL = {a: i for i, a in enumerate(ACTIONS)}
IDLE = LABEL.get("_idle")
MIRROR_PERM = np.arange(33)
for a, b in [(1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16), (17, 18), (19, 20), (21, 22),
             (23, 24), (25, 26), (27, 28), (29, 30), (31, 32)]:
    MIRROR_PERM[a], MIRROR_PERM[b] = b, a

for c in data:
    s, e = c["seg"]
    seq = c["raw"][s:e]
    a0, a1 = c["act"][0] - s, c["act"][1] - s
    if len(seq) < WINDOW:
        left = (WINDOW - len(seq)) // 2
        seq = np.pad(seq, ((left, WINDOW - len(seq) - left), (0, 0)), mode="edge")
        a0, a1 = a0 + left, a1 + left
    c["seq"], c["sa"] = seq, (max(a0, 0), min(a1, len(seq)))
    idle_idx = np.r_[np.arange(c["sa"][1] + 2, len(seq)), np.arange(0, max(c["sa"][0] - 2, 0))]
    c["idle_idx"] = idle_idx if c["act_found"] and len(idle_idx) >= 8 else None
    c["label"] = LABEL[c["word"]]

def good_starts(n_frames, act, other=None):
    starts = np.arange(0, n_frames - WINDOW + 1)
    a0, a1 = act
    ov = np.clip(np.minimum(starts + WINDOW, a1) - np.maximum(starts, a0), 0, None)
    good = (ov >= LABEL_COVERAGE * (a1 - a0)) | (ov >= LABEL_COVERAGE * WINDOW)
    if other is not None:
        ov2 = np.clip(np.minimum(starts + WINDOW, other[1]) - np.maximum(starts, other[0]), 0, None)
        good &= ov2 >= 4
    return starts[good] if good.any() else starts[[int(np.argmax(ov))]]

for c in data:
    c["starts"] = good_starts(len(c["seq"]), c["sa"])

def idle_window(c, r):
    idx = c["idle_idx"]
    pp = np.r_[idx, idx[::-1]]
    o = r.integers(len(pp))
    return c["seq"][pp[(o + np.arange(WINDOW)) % len(pp)]]

def transition_window(A, B, r):
    """End of sign A running straight into sign B (no rest), labelled with whichever sign fills the window."""
    a = A["seq"][:A["sa"][1]]
    b0 = max(0, B["sa"][0] - 2)
    seq = np.concatenate([a, B["seq"][b0:]])
    act_a, act_b = (A["sa"][0], len(a)), (len(a) + B["sa"][0] - b0, len(a) + B["sa"][1] - b0)
    if len(seq) < WINDOW:
        seq = np.pad(seq, ((0, WINDOW - len(seq)), (0, 0)), mode="edge")
    if r.random() < 0.5:
        target, other, label = act_b, act_a, B["label"]
    else:
        target, other, label = act_a, act_b, A["label"]
    s = r.choice(good_starts(len(seq), target, other))
    return seq[s:s + WINDOW], label

def augment(win, aspect, word, r):
    W = len(win)
    if AUG_SPEED:
        f, mid = r.uniform(1 - AUG_SPEED, 1 + AUG_SPEED), (W - 1) / 2
        win = win[np.clip(np.round(mid + (np.arange(W) - mid) * f), 0, W - 1).astype(int)]
    pose = win[:, F.POSE_SLICE].reshape(W, 33, 4).copy()
    ha = win[:, F.HAND_A_SLICE].reshape(W, 21, 3).copy()
    hb = win[:, F.HAND_B_SLICE].reshape(W, 21, 3).copy()
    th = np.deg2rad(r.uniform(-AUG_ROTATE_DEG, AUG_ROTATE_DEG))
    rot = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    scale, squash = r.uniform(1 - AUG_SCALE, 1 + AUG_SCALE), r.uniform(1 - AUG_SQUASH, 1)
    shift = r.uniform(-AUG_SHIFT, AUG_SHIFT, 2)
    mirror = word not in NO_MIRROR_WORDS and r.random() < MIRROR_PROB
    centre, asp = np.array([0.5 * aspect, 0.5]), np.array([aspect, 1.0])
    for pts in (pose, ha, hb):
        present = np.any(pts != 0, axis=(1, 2))
        xy = pts[..., :2] * asp - centre
        if mirror:
            xy[..., 0] = -xy[..., 0]
        xy[..., 0] *= squash
        xy = (xy @ rot.T) * scale + centre + shift + r.normal(0, AUG_NOISE, xy.shape)
        pts[..., :2] = np.where(present[:, None, None], xy / asp, 0)
        pts[..., 2] *= scale
    if mirror:
        pose, ha, hb = pose[:, MIRROR_PERM], hb, ha
    if r.random() < LOWER_BODY_DROPOUT:
        cut = r.choice([23, 25, 27])
        pose[:, cut:, 3] = np.where(np.any(pose != 0, axis=(1, 2))[:, None], r.uniform(0, 0.3, (W, 33 - cut)), 0)
    return np.concatenate([pose.reshape(W, -1), ha.reshape(W, -1), hb.reshape(W, -1)], axis=1)

train = [c for c in data if c["split"] == "train"]
partners = defaultdict(list)
for i, c in enumerate(train):
    partners[(c["group"] if c["source"] == "online" else c["person"], round(c["aspect"], 2))].append(i)
specs = [("word", i, None) for i, c in enumerate(train) for _ in range(WINDOWS_PER_CLIP)]
n_word = len(specs)
idle_ok = [i for i, c in enumerate(train) if c["idle_idx"] is not None]
if IDLE_CLASS and idle_ok:
    n_idle = max(1, n_word // len(usable_words))
    specs += [("idle", int(i), None) for i in rng.choice(idle_ok, n_idle)]
trans_ok = [i for i, c in enumerate(train)
            if len({train[j]["word"] for j in partners[(c["group"] if c["source"] == "online" else c["person"],
                                                         round(c["aspect"], 2))]}) > 1]
if TRANSITIONS and trans_ok:
    specs += [("trans", int(i), None) for i in rng.choice(trans_ok, int(n_word * TRANSITION_SHARE))]
print(f"Training windows per pass: {n_word} sign + {sum(s[0] == 'idle' for s in specs)} idle + "
      f"{sum(s[0] == 'trans' for s in specs)} sign-to-sign transitions "
      f"(from {len(train)} clips; idle uses {len(idle_ok)} clips with a rest pose).")

class Windows(tf.keras.utils.Sequence):
    """Makes each batch fresh: random window position, random augmentation, then the v2 normalisation."""
    def __init__(self, specs, batch, seed):
        super().__init__()
        self.specs, self.batch, self.r = list(specs), batch, np.random.default_rng(seed)
        self.r.shuffle(self.specs)

    def __len__(self):
        return math.ceil(len(self.specs) / self.batch)

    def make(self, spec):
        kind, i, _ = spec
        c = train[i]
        if kind == "idle":
            win, label, word = idle_window(c, self.r), IDLE, c["word"]
        elif kind == "trans":
            key = (c["group"] if c["source"] == "online" else c["person"], round(c["aspect"], 2))
            j = self.r.choice([j for j in partners[key] if train[j]["word"] != c["word"]])
            win, label = transition_window(c, train[j], self.r)
            word = ACTIONS[label]
            if word in NO_MIRROR_WORDS or c["word"] in NO_MIRROR_WORDS or train[j]["word"] in NO_MIRROR_WORDS:
                word = NO_MIRROR_WORDS[0]
        else:
            s = self.r.choice(c["starts"])
            win, label, word = c["seq"][s:s + WINDOW], c["label"], c["word"]
        return F.normalize(augment(win, c["aspect"], word, self.r), c["aspect"]), label

    def __getitem__(self, k):
        out = [self.make(s) for s in self.specs[k * self.batch:(k + 1) * self.batch]]
        return np.stack([o[0] for o in out]), np.array([o[1] for o in out], np.int32)

    def on_epoch_end(self):
        self.r.shuffle(self.specs)

def fixed_windows(cs, per_clip=WINDOWS_PER_CLIP, idle=True, seed=0):
    """Validation/test windows: evenly spaced, no augmentation."""
    r = np.random.default_rng(seed)
    X, y, owner = [], [], []
    for n, c in enumerate(cs):
        st = c["starts"]
        for s in np.unique(st[np.linspace(0, len(st) - 1, min(per_clip, len(st))).round().astype(int)]):
            X.append(F.normalize(c["seq"][s:s + WINDOW], c["aspect"])); y.append(c["label"]); owner.append(n)
        if idle and IDLE is not None and c["idle_idx"] is not None:
            X.append(F.normalize(idle_window(c, r), c["aspect"])); y.append(IDLE); owner.append(-1)
    return np.asarray(X, np.float32), np.asarray(y, np.int32), np.asarray(owner)

val_clips = [c for c in data if c["split"] == "val"]
test_clips = [c for c in data if c["split"] == "test"]
X_val, y_val, _ = fixed_windows(val_clips)
train_gen = Windows(specs, BATCH, SEED)
xb, yb = train_gen[0]
print(f"Validation: {len(X_val)} windows from {len(val_clips)} clips. Batch shape {xb.shape}, "
      f"feature range {xb.min():.1f}..{xb.max():.1f}.")

In [ ]:
#@title 8. Train
labels_all = np.array([train_gen.make(s)[1] if s[0] == "trans" else (IDLE if s[0] == "idle" else train[s[1]]["label"])
                       for s in specs])
counts = np.bincount(labels_all, minlength=len(ACTIONS)).astype(float)
class_weight = {i: float(len(labels_all) / (len(ACTIONS) * max(n, 1))) for i, n in enumerate(counts)}

L = tf.keras.layers
model = tf.keras.Sequential([
    L.Input((WINDOW, F.FEATURE_DIM)),
    L.LSTM(128, return_sequences=True), L.Dropout(0.3),
    L.LSTM(128), L.Dropout(0.3),
    L.Dense(128, activation="relu"), L.Dropout(0.3),
    L.Dense(len(ACTIONS), activation="softmax"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

class EpochLog(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.t, self.best = time.time(), 0.0
    def on_epoch_end(self, epoch, logs=None):
        self.best = max(self.best, logs["val_accuracy"])
        lr = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
        print(f"epoch {epoch + 1:3d} | train acc {logs['accuracy']:.3f} loss {logs['loss']:.3f} | "
              f"val acc {logs['val_accuracy']:.3f} loss {logs['val_loss']:.3f} | best {self.best:.3f} | "
              f"lr {lr:.0e} | {(time.time() - self.t) / 60:.1f} min", flush=True)

ckpt = str(OUT_DIR / "best_during_training.h5")
history = model.fit(
    train_gen, validation_data=(X_val, y_val), epochs=EPOCHS, class_weight=class_weight, verbose=0,
    callbacks=[EpochLog(),
               tf.keras.callbacks.EarlyStopping("val_loss", patience=PATIENCE, restore_best_weights=True),
               tf.keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=max(PATIENCE // 3, 3), min_lr=1e-5),
               tf.keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True)])
EPOCHS_RUN = len(history.history["loss"])
VAL_ACC = float(model.evaluate(X_val, y_val, verbose=0)[1])
print(f"\nStopped after {EPOCHS_RUN} epochs. Validation accuracy (best weights): {VAL_ACC:.3f}")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, k in zip(axes, ("accuracy", "loss")):
    ax.plot(history.history[k], label="train"); ax.plot(history.history["val_" + k], label="validation")
    ax.set_title(k); ax.legend()
plt.show()

In [ ]:
#@title 9. Test on clips the model never saw
def predict(X):
    return model.predict(X, batch_size=512, verbose=0) if len(X) else np.zeros((0, len(ACTIONS)))

def clip_results(cs, transform=None):
    """One prediction per clip: average of its windows (sign windows only)."""
    if transform:
        cs = [dict(c, seq=transform(c["seq"], c["aspect"])) for c in cs]
    X, y, owner = fixed_windows(cs, idle=False)
    P = predict(X)
    preds = np.array([P[owner == n].mean(0).argmax() if (owner == n).any() else -1 for n in range(len(cs))])
    return preds, np.array([c["label"] for c in cs])

def acc(p, y):
    return float((p == y).mean()) if len(y) else float("nan")

preds, truth = clip_results(test_clips)
src = np.array([c["source"] for c in test_clips])
X_t, y_t, own_t = fixed_windows(test_clips)
P_t = predict(X_t)
win_acc = acc(P_t.argmax(1)[own_t >= 0], y_t[own_t >= 0])
idle_acc = acc(P_t.argmax(1)[own_t < 0], y_t[own_t < 0])
TEST = {"clips": acc(preds, truth), "team": acc(preds[src == "team"], truth[src == "team"]),
        "online": acc(preds[src == "online"], truth[src == "online"]), "windows": win_acc, "idle": idle_acc}
print(f"TEST ({len(test_clips)} clips): {TEST['clips']:.1%} of clips right | team {TEST['team']:.1%} | "
      f"online (other people, other cameras) {TEST['online']:.1%} | single windows {win_acc:.1%} | "
      f"rest windows recognised as _idle {idle_acc:.1%}")

print(f"\n{'word':16}{'team test':>12}{'online test':>13}   most often mistaken for")
confusions = Counter()
for p, t in zip(preds, truth):
    if p != t and p >= 0:
        confusions[(ACTIONS[t], ACTIONS[p])] += 1
suspect = []
for w in usable_words:
    i = LABEL[w]
    m_t, m_o = (truth == i) & (src == "team"), (truth == i) & (src == "online")
    a_t, a_o = acc(preds[m_t], truth[m_t]), acc(preds[m_o], truth[m_o])
    wrong = Counter({k[1]: v for k, v in confusions.items() if k[0] == w}).most_common(2)
    print(f"{w:16}{a_t:>9.0%} ({m_t.sum()}){a_o:>9.0%} ({m_o.sum()})   {', '.join(f'{x} x{n}' for x, n in wrong)}")
    if m_o.sum() >= 3 and a_t >= 0.9 and a_o < 0.5:
        suspect.append(w)
if suspect:
    print(f"\nWatch out: {', '.join(suspect)} are right for the team signer but mostly wrong for other people. The model "
          "may be recognising the person, not the sign. More signers for these words would fix it.")

words_only = [LABEL[w] for w in usable_words]
cm = np.zeros((len(words_only), len(words_only)), int)
for p, t in zip(preds, truth):
    if p in words_only:
        cm[words_only.index(t), words_only.index(p)] += 1
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(cm / np.maximum(cm.sum(1, keepdims=True), 1), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(words_only))); ax.set_xticklabels(usable_words, rotation=90, fontsize=8)
ax.set_yticks(range(len(words_only))); ax.set_yticklabels(usable_words, fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Test clips (rows = true word)")
plt.tight_layout(); plt.show()

def reframe(zoom=1.0, rotate=0.0, shift=(0.0, 0.0), upper_body=False):
    """Pretend the camera is placed differently. Points pushed out of the picture disappear, like in real life."""
    def f(seq, aspect):
        out = seq.copy()
        T = len(out)
        th = np.deg2rad(rotate)
        rot = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
        centre, asp = np.array([0.5 * aspect, 0.5]), np.array([aspect, 1.0])
        for sl, n, dim in ((F.POSE_SLICE, 33, 4), (F.HAND_A_SLICE, 21, 3), (F.HAND_B_SLICE, 21, 3)):
            pts = out[:, sl].reshape(T, n, dim)
            present = np.any(pts != 0, axis=(1, 2))
            xy = ((pts[..., :2] * asp - centre) @ rot.T) * zoom + centre + np.array(shift)
            pts[..., :2] = np.where(present[:, None, None], xy / asp, 0)
            pts[..., 2] *= zoom
            inside = (pts[..., 0] >= 0) & (pts[..., 0] <= 1) & (pts[..., 1] >= 0) & (pts[..., 1] <= 1)
            if dim == 4:
                pts[..., 3] = np.where(inside, pts[..., 3], 0.0)
                if upper_body:
                    pts[:, 23:, 3] = np.minimum(pts[:, 23:, 3], 0.1)
            else:
                pts[inside.sum(1) < 15] = 0
            out[:, sl] = pts.reshape(T, -1)
        return out
    return f

print("\nCamera placement test (same test clips, camera moved):")
FRAMING = {}
for name, tfm in [("as recorded", None), ("farther away (person 70% size)", reframe(0.7)),
                  ("closer (130%)", reframe(1.3)), ("upper body only (legs unseen)", reframe(upper_body=True)),
                  ("tilted 10 degrees", reframe(rotate=10)), ("person off to the side", reframe(shift=(0.15, 0.0)))]:
    p, t = clip_results(test_clips, tfm)
    FRAMING[name] = acc(p, t)
    print(f"  {name:34} {FRAMING[name]:.1%}")

def replay(raw_seq, aspect):
    """Runs the Jetson's exact rules (window, hand gate, STABLE_N, THRESHOLD, _idle, repeat) over a stream."""
    feats = F.normalize(raw_seq, aspect)
    hands = np.any(raw_seq[:, F.HANDS_SLICE] != 0, axis=1)
    T = len(feats)
    if T < WINDOW:
        return []
    P = predict(np.stack([feats[e - WINDOW:e] for e in range(WINDOW, T + 1)]))
    emitted, recent, last, idle = [], [], None, 0
    def note_idle():
        nonlocal idle, last
        if REPEAT_RESET > 0:
            idle += 1
            if idle >= REPEAT_RESET:
                last = None
    for e in range(WINDOW, T + 1):
        if hands[e - WINDOW:e].mean() < MIN_HAND_FRAC:
            recent = []; note_idle(); continue
        p = P[e - WINDOW]
        recent = (recent + [int(p.argmax())])[-STABLE_N:]
        if len(recent) < STABLE_N or len(set(recent)) != 1 or p.max() < THRESHOLD:
            note_idle(); continue
        g = ACTIONS[recent[-1]]
        if g.startswith("_"):
            last, idle = None, 0; continue
        if g != last:
            emitted.append(g)
        last, idle = g, 0
    return emitted

def hold(seq, before=8, after=12):
    return np.concatenate([np.repeat(seq[:1], before, 0), seq, np.repeat(seq[-1:], after, 0)])

right = nothing = wrong = 0
for c in test_clips:
    out = replay(hold(c["seq"]), c["aspect"])
    if not out:
        nothing += 1
    elif out[0] == c["word"] and len(out) == 1:
        right += 1
    else:
        wrong += 1
n = max(len(test_clips), 1)
REPLAY = {"right": right / n, "nothing": nothing / n, "wrong_or_extra": wrong / n}
print(f"\nJetson replay, one clip at a time (threshold {THRESHOLD}, {STABLE_N} agreeing windows): "
      f"{right / n:.1%} exactly right | {nothing / n:.1%} said nothing | {wrong / n:.1%} wrong or extra words")

def lcs(a, b):
    m = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a)):
        for j in range(len(b)):
            m[i + 1][j + 1] = m[i][j] + 1 if a[i] == b[j] else max(m[i][j + 1], m[i + 1][j])
    return m[-1][-1]

r_chain = random.Random(SEED)
by_person = defaultdict(list)
for c in test_clips:
    if c["source"] == "team" and c["act_found"]:
        by_person[c["person"]].append(c)
CHAINS = {}
for pause in (0, 10):
    exp_n = hit = extra = 0
    for person, cs in by_person.items():
        for _ in range(15):
            pick, used = [], set()
            for c in r_chain.sample(cs, len(cs)):
                if c["word"] not in used:
                    pick.append(c); used.add(c["word"])
                if len(pick) == 4:
                    break
            if len(pick) < 2:
                continue
            parts = []
            for k, c in enumerate(pick):
                s0 = 0 if k == 0 else max(c["sa"][0] - 2, 0)
                s1 = len(c["seq"]) if k == len(pick) - 1 else c["sa"][1]
                parts.append(c["seq"][s0:s1])
                if pause and k < len(pick) - 1:
                    parts.append(np.repeat(c["seq"][s1 - 1:s1], pause, 0))
            out = replay(hold(np.concatenate(parts)), pick[0]["aspect"])
            expected = [c["word"] for c in pick]
            h = lcs(expected, out)
            exp_n, hit, extra = exp_n + len(expected), hit + h, extra + len(out) - h
    CHAINS[pause] = (hit / max(exp_n, 1), extra / max(exp_n, 1))
    label = "signs back to back, no pause" if pause == 0 else f"{pause / SAMPLE_FPS:.1f} s pause between signs"
    print(f"Sentences of 2-4 team test clips ({label}): {hit / max(exp_n, 1):.1%} of words caught in order, "
          f"{extra / max(exp_n, 1):.2f} extra words per expected word")

In [ ]:
#@title 10. Save the model (+ copy to the Jetson folder if COPY_TO_JETSON)
import datetime, keras
model_file = OUT_DIR / f"{MODEL_NAME}.h5"
model.save(model_file, save_format="h5")
meta = {
    "model_file": model_file.name,
    "actions": ACTIONS,
    "window": WINDOW,
    "frame_stride": 2,
    "capture_fps": 30,
    "effective_fps": float(SAMPLE_FPS),
    "feature_dim": F.FEATURE_DIM,
    "feature_version": F.FEATURE_VERSION,
    "min_hand_frac": MIN_HAND_FRAC,
    "stable_n": STABLE_N,
    "threshold": THRESHOLD,
    "repeat_reset": REPEAT_RESET,
    "frame_size": FRAME_SIZE,
    "model_complexity": 0,
    "versions": {"tensorflow": tf.__version__, "keras": keras.__version__, "mediapipe": mp.__version__,
                 "numpy": np.__version__},
    "clips_used": {w: sum(c["word"] == w for c in data) for w in usable_words},
    "hand_check": {"active_usable_min": ACTIVE_USABLE_MIN, "clip_usable_min": CLIP_USABLE_MIN,
                   "hand_points_inside": HAND_POINTS_INSIDE, "body_points_min": BODY_POINTS_MIN,
                   "kept": tot["keep"], "rescued": tot["rescue"], "dropped": tot["drop"]},
    "epochs_run": EPOCHS_RUN,
    "validation_accuracy": VAL_ACC,
    "test": TEST,
    "framing_test": FRAMING,
    "jetson_replay": REPLAY,
    "trained_utc": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
}
(OUT_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2))
shutil.copy2(JETSON_DIR / "asl_features.py", OUT_DIR / "asl_features.py")

check = tf.keras.models.load_model(model_file)
assert check.input_shape == (None, WINDOW, F.FEATURE_DIM) and check.output_shape == (None, len(ACTIONS))
print(f"Saved to {OUT_DIR}:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:28} {f.stat().st_size / 1e6:.2f} MB")

if COPY_TO_JETSON:
    dest = JETSON_DIR / "models" / MODEL_NAME
    dest.mkdir(parents=True, exist_ok=True)
    for name in (model_file.name, "model_meta.json"):
        shutil.copy2(OUT_DIR / name, dest / name)
    print(f"Copied to {dest}")
print(f"\nOn the Jetson: MODEL_PATH=models/{MODEL_NAME}/{MODEL_NAME}.h5 "
      "(the word list, window and thresholds come from model_meta.json).")
if IN_COLAB:
    from google.colab import files
    shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
    files.download(str(OUT_DIR) + ".zip")